# RQ1 — effort-reduction curve under a forest-only background (robustness check)

**Purpose.** `effort_reduction_curve.ipynb` found Random Forest and
Logistic Regression already near-ceiling at 0% expert data (Boyce
≈0.99), flat up to 90%, only dropping at 100%. Before accepting "no
partial amount of expert data earns a visible benefit" as a finding, I
checked whether this is a bug (it is not — `src/evaluation.py` is
correct, there is no train/test leakage, see chat) and whether it is a
background-composition artifact: the shared background pool (2 000
uniform random points) is ~29% non-forest land cover (roads,
agriculture, water), while the acacia presences are ~92% forest, so
part of the apparent separability may just be "forest vs. not forest,"
not "acacia vs. other forest" — an easier problem that a harder,
habitat-matched background would not inflate.

This check re-runs the full curve against a **forest-only background
pool**: same construction as
[`one_class_method_selection.ipynb`](one_class_method_selection.ipynb)'s
shared pool (uniform random points in the pilot bbox, 50 m exclusion
buffer around every known acacia presence), with one added filter —
a point only counts if it falls inside a COS 2023 **forest-type**
polygon: eucalyptus, pine, "other forest," or shrubland. Points in
artificial, agriculture, water/bare **and acacia-forest (COS)** land
cover are excluded — the last one because keeping a COS-labelled
"acacia forest" point as a "not acacia" background point would be a
labelling contradiction, not a harder test.

**Status: exploratory / not yet a design decision.** Nothing here
replaces the shared background pool used elsewhere in RQ1 — this is a
one-off robustness check on the effort-reduction-curve finding, run at
my own initiative in response to a plausibility question, not yet
confirmed with the supervisor either way.

**Where it is computed.** The curve is trained and scored by
[`experiments/effort_reduction_curve/run.py`](../../experiments/effort_reduction_curve/run.py),
the script of the main curve, run with `--background forest_only`; it
writes to `results/effort_reduction_curve_forest_only/`. The
uniform-background side of the comparison is the run of the main
curve in `results/effort_reduction_curve/`. The only difference
between the two runs is the background pool. This notebook only reads
the two runs, draws the figure and holds the reading. Current values:
Section 3; the text of Section 5 was written on 2026-11-07 and has a
note with the current values.


> **Question numbering (note of 2026-11-13).** Current numbering in
> `docs/research_questions.md`: RQ1 = effort-reduction curve (citizen
> science against expert data); RQ2 = from points to reference data
> (segmentation mask, grade and accuracy filters); RQ3 =
> transferability away from the sampled area; RQ4 = TESSERA against
> Sentinel-2.
> In this notebook "RQ1" is the effort-reduction curve, which is still
> RQ1.

## 1. Load the forest-only background pool

Same rejection-sampling pipeline as the shared background pool, plus a
COS 2023 land-cover filter. COS is read bbox-chunked per 10 km block
(same pattern as `expert_polygons.ipynb`'s `cos_group` classification),
applied to each batch of candidates that already survived the
exclusion buffer. Target: N = 2 000, same as the existing pool, so this
is a clean swap-in comparison rather than confounding pool size with
background difficulty. Saved separately
(`background_points_forest_only.npz`) — the original pool is untouched
and still used everywhere else in RQ1.

The pool is built in
[`background_pools.ipynb`](../05_data_integration/background_pools.ipynb)
(`src.background.sample_forest_only`); this notebook only reads it,
after checking it against `data/processed_manifest.csv`.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src import manifest
from src.background import load_background

# Built in notebooks/05_data_integration/background_pools.ipynb; load_background() checks the
# file and its inputs against data/processed_manifest.csv before loading.
bgf = load_background(ROOT, "forest_only")
print(f"{len(bgf['emb']):,} forest-only background points with valid embeddings")
print(pd.Series(bgf["cos_group"]).value_counts())

## 2. Load the two saved runs

Both runs come from the same script, with the same seed-0 polygon
order and the same test blocks per seed (the test blocks are drawn
from the citizen-science ∪ expert presence blocks, not from the
background). The hashes of the input files recorded in each
`run.json` are compared with the current files: if one differs, the
script has to be run again.

In [ ]:
RUNS = {"full": ROOT / "results" / "effort_reduction_curve",
        "forest_only": ROOT / "results" / "effort_reduction_curve_forest_only"}

run_info = {}
for label, out_dir in RUNS.items():
    run = json.loads((out_dir / "run.json").read_text())
    run_info[label] = run
    changed = [p for p, h in run["inputs_sha256"].items() if manifest.sha256_path(ROOT / p) != h]
    if changed:
        warnings.warn(f"{label}: these inputs changed since the run of {run['run_date']}: {changed}. "
                      f"Run again: {run['command']}")
    else:
        print(f"{label}: run of {run['run_date']} ({run['command']}), background '{run['background']}': "
              "the input files are the current ones")

run = run_info["forest_only"]
print(f"citizen-science presences: {run['n_cs_presences']:,} (matched {run['n_cs_matched']}/{run['n_cs_rows']})")
print(f"expert presence pixels: {run['n_expert_pixels']:,}, across {run['n_expert_polygons']:,} polygons")
print(f"{run['n_candidate_blocks']} candidate blocks (citizen-science ∪ expert presences)")
pd.read_csv(RUNS["forest_only"] / "test_sizes.csv")[["seed", "n_test_cs", "n_test_expert_pixels", "n_test_background"]]

## 3. Aggregate and compare against the uniform (full) background curve

Scores on the pooled test presences (citizen science and expert
together), mean and standard deviation over the 5 seeds. Random
Forest and Logistic Regression come from `summary.csv`; the One-Class
SVM from `one_class_per_seed.csv`.

In [ ]:
COLS = ["classifier", "level_pct", "roc_auc_mean", "roc_auc_sd", "boyce_mean", "boyce_sd"]

def pooled_summary(out_dir, label):
    pb = pd.read_csv(out_dir / "summary.csv")
    pb = pb[pb["test_presences"] == "pooled"][COLS]
    oc = (
        pd.read_csv(out_dir / "one_class_per_seed.csv")
        .groupby(["classifier", "level_pct"])
        .agg(roc_auc_mean=("roc_auc", "mean"), roc_auc_sd=("roc_auc", "std"),
             boyce_mean=("boyce", "mean"), boyce_sd=("boyce", "std"))
        .reset_index()
    )
    out = pd.concat([pb, oc]).sort_values(["classifier", "level_pct"]).reset_index(drop=True)
    out["background"] = label
    return out

summary_forest = pooled_summary(RUNS["forest_only"], "forest_only")
summary_forest.round(3)

Uniform (full, 2 000-point) background: the run of the main curve,
read from `results/effort_reduction_curve/`. `drop` = full minus
forest-only.

In [ ]:
summary_full = pooled_summary(RUNS["full"], "full")

compare = pd.concat([summary_full, summary_forest], ignore_index=True)
pivot = compare.pivot_table(index=["classifier", "level_pct"], columns="background", values=["roc_auc_mean", "boyce_mean"])
pivot["roc_auc_mean", "drop"] = pivot["roc_auc_mean", "full"] - pivot["roc_auc_mean", "forest_only"]
pivot["boyce_mean", "drop"] = pivot["boyce_mean", "full"] - pivot["boyce_mean", "forest_only"]
pivot.round(3)

## 4. Plot: full vs. forest-only background, by classifier

Same two-panel layout (ROC-AUC, Boyce) and fixed classifier colours as
`effort_reduction_curve.ipynb`; the forest-only run is drawn with a
dashed line of the same colour, so each classifier's own full-vs-forest
gap is easy to read without crowding the legend.


In [ ]:
import matplotlib.pyplot as plt

COLORS = {"OneClassSVM": "#2a78d6", "RandomForest": "#eb6834", "LogisticRegression": "#1baf7a"}
MARKERS = {"OneClassSVM": "o", "RandomForest": "s", "LogisticRegression": "^"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)

for metric, ax, title in [("roc_auc_mean", axes[0], "ROC-AUC"), ("boyce_mean", axes[1], "Continuous Boyce index")]:
    for clf, color in COLORS.items():
        for bgname, ls, alpha in [("full", "-", 1.0), ("forest_only", "--", 0.85)]:
            sub = compare[(compare["classifier"] == clf) & (compare["background"] == bgname)].sort_values("level_pct")
            ax.plot(sub["level_pct"], sub[metric], color=color, marker=MARKERS[clf], linestyle=ls,
                    markersize=5, linewidth=2, alpha=alpha,
                    label=f"{clf} ({bgname})" if metric == "roc_auc_mean" else None)
    ax.set_title(title)
    ax.set_xlabel("% of expert polygons added to citizen-science base")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.8, zorder=0)

axes[0].set_ylabel("score (mean over 5 seeds)")
axes[0].legend(frameon=False, loc="lower right", fontsize=8)
fig.suptitle("RQ1 effort-reduction curve — full vs. forest-only background", y=1.02)
fig.tight_layout()
plt.savefig(RUNS["forest_only"] / "curve_full_vs_forest_only.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Reading the comparison (2026-11-07)

> **Note of 2026-11-13.** The numbers in the text below are from the
> run of 2026-11-07, made before the presences were restricted to
> 2021–2025 and the uniform background was rebuilt. Current values
> (Section 3, runs of 2026-11-13), pooled test presences:
>
> - **Random Forest.** ROC-AUC is higher under the forest-only
>   background at every level (0.902 → 0.950 at 0%, 0.906 → 0.952 at
>   90%, 0.854 → 0.919 at 100%) and flat from 0% to 90% under both.
>   Boyce is 0.98–1.00 from 0% to 90% under both; at 100% it is 0.969
>   (full) and 0.902 (forest-only).
> - **Logistic Regression.** ROC-AUC 0.893 → 0.958 at 0%, 0.924 →
>   0.962 at 90%, 0.873 → 0.905 at 100%. Boyce at 100%: 0.841 (full)
>   against 0.727 (forest-only, SD 0.281 over the seeds); the value
>   0.567 in the text is not the current one.
> - **One-Class SVM.** ROC-AUC at 0%: 0.574 (full) against 0.504
>   (forest-only); at 100%: 0.831 against 0.814. Boyce at 0%: 0.668
>   against 0.197 (SD 0.723), so it is not negative in the current
>   run; from 10% to 90% it is 0.88–0.93 (full) against 0.46–0.66
>   (forest-only).
>
> The direction of each statement below is the same with the current
> values; the sizes are not.

**The "flat curve, no visible benefit from partial expert data" finding
survives a much harder background — and for Random Forest it holds up
slightly better, not worse.**

- **Random Forest.** ROC-AUC actually *rose* under the forest-only
  background at every mixing level (e.g. 0.879 → 0.934 at 0%, 0.878 →
  0.933 at 90%), and the curve stayed just as flat across 0–90% as
  under the full background. Continuous Boyce stayed ~0.98–0.99 under
  both. At 100% (expert-only training, no citizen science), ROC-AUC
  also rose (0.801 → 0.885) and Boyce dropped only marginally (0.917 →
  0.905). **Restricting the negative class to ecologically matched
  forest types (eucalyptus, pine, other forest, shrubland — excluding
  the COS-labelled acacia class as a labelling contradiction) does not
  expose any hidden weakness.** The near-ceiling score and the flat
  shape are not an artifact of an easy, non-forest background.
- **Logistic Regression.** Same overall pattern (ROC-AUC up ~0.04–0.06
  across 0–90%, flat shape preserved), with one exception worth
  flagging: at the 100% (expert-only) point its Boyce index collapses
  from 0.897 (full background) to 0.567 (forest-only), even though
  ROC-AUC there still improves (0.822 → 0.870). Ranking quality (AUC)
  stays fine; the Boyce index — which is threshold/density-sensitive —
  does not, specifically at the smallest, most extreme training-set
  point. Flagged as a caveat, not folded into the headline conclusion.
- **One-Class SVM behaves very differently.** ROC-AUC collapses to
  near chance at low mixing levels (0.515 at 0%, vs. 0.574 under the
  full background) and Boyce goes **negative** at 0% (-0.039, vs. 0.628
  under the full background) — the presence-only decision boundary is
  barely better than noise at telling acacia apart from other forest
  types once the "easy" non-forest background is removed. This
  strongly suggests One-Class SVM's earlier, more moderate scores
  (ROC-AUC 0.57–0.77 across the curve) were substantially inflated by
  the background's non-forest component — it was partly succeeding by
  recognising "this clearly isn't vegetation" rather than learning
  anything acacia-specific, exactly the risk informally expected of a
  presence-only method without negative supervision. The monotonic
  *improvement* with more expert data is still present (ROC-AUC
  0.515→0.754 from 0% to 100%), actually larger in relative terms, but
  off a much lower floor.

**Answer to "couldn't it be a mistake?" (the question that motivated
this whole check):** no — across the code review of
`continuous_boyce()`, the leakage distance check, the land-cover
composition comparison, and now this forest-only robustness re-run of
the *entire* curve, the Random Forest / Logistic Regression
near-ceiling result and the "no partial benefit from expert data"
reading hold up. The one real update: One-Class SVM's apparent partial
competitiveness earlier in the curve now looks like it was
substantially a background-composition artifact specific to that one
model — not something that touches the presence-vs-background
classifiers (Random Forest, proposed default for RQ1).

**Caveats, carried over from the main curve notebook plus new ones
specific to this check:**
- Forest-only background is still N=2,000 (matched to the original
  pool's size, not independently tuned or enlarged).
- The forest-type COS grouping (eucalyptus / pine / other forest /
  shrubland) is my own grouping of COS 2023 codes, not yet confirmed
  by the supervisor.
- The Logistic Regression Boyce collapse at 100% is noted but not
  further investigated here.
- This is a one-off robustness check layered on top of an
  already-unconfirmed shared background-pool design
  (`data/README.md`) — it does not replace that pool, only stress-tests
  the curve's conclusion against a harder alternative.
- No hyperparameter re-tuning for either background; same
  configurations as every other RQ1 notebook so far.
